# F

Appelsap

## Imports & Data inladen

In [21]:
import pandas as pd
import numpy as np
import time
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout

In [ ]:
BASE_PATH = '../MLOps Data/'

transaction_train = pd.read_csv(BASE_PATH + 'train_transaction.csv')
transaction_test = pd.read_csv(BASE_PATH + 'test_transaction.csv')

identity_train = pd.read_csv(BASE_PATH + 'train_identity.csv')
identity_test = pd.read_csv(BASE_PATH + 'test_identity.csv')

In [23]:
train = pd.merge(transaction_train, identity_train, on='TransactionID', how='left')
test = pd.merge(transaction_test, identity_test, on='TransactionID', how='left')

# Pipeline via Class

In [ ]:
# zet alle functies in deze class, hatseflats gelijk pipeline

class dataPipelineDinges:
    def __init__(self, train, test):
        self.df_train = train
        self.df_test = test

        self.X_train = self.df_train.drop(['isFraud'], axis=1)
        self.y_train = self.df_train['isFraud']

        # features volgens copilot, weet niet hoe goed deze zijn
        self.features = ['TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 
                        'card5', 'addr1', 'addr2', 'dist1', 'dist2']


    def process_data():
        # eda
        # data cleaning
        # feature engineering
        None


    def edge_model(self):   
        # edge_duration = time.time()
        edge_model = IsolationForest(n_estimators=50, contamination=0.01, random_state=42)
        edge_model.fit(self.X_train[self.features])  # edge_features nog achter X_train, maar weet niet of die features wel kloppen
        # edge_duration = time.time() - edge_duration
        # print(f"Edge model trained in {edge_duration:.2f} seconds")

        # Sla het model op
        joblib.dump(edge_model, 'edge_model.pkl')
        return edge_model
    

    def cloud_model(self):    
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(self.X_train[self.features])

        # cloud_duration = time.time()
        cloud_model = Sequential([
            Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
            Dropout(0.3),
            Dense(32, activation='relu'),
            Dropout(0.2),
            Dense(1, activation='sigmoid')
        ])

        cloud_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

        # Train the model
        cloud_model.fit(X_train_scaled, self.y_train, epochs=10, batch_size=256)
        # cloud_duration = time.time() - cloud_duration
        # print(f"Cloud model trained in {cloud_duration:.2f} seconds")
        # Sla het model op
        cloud_model.save('cloud_model.h5')

        return cloud_model
    
    def statistics(self):
        # geef statistieken over performance van modellen

        # runnen als deel van predict?
        None

    def predict(self, model, data):
        if isinstance(model, IsolationForest):
            return model.predict(data)
        elif isinstance(model, Sequential):
            data_scaled = StandardScaler().fit_transform(data)
            return model.predict(data_scaled)
        else:
            raise ValueError("Unsupported model type")


    def run(self):
        # self.process_data()
        edge_model = self.edge_model()
        cloud_model = self.cloud_model()
        self.statistics()
        edge_predictions = self.predict(edge_model, self.X_train[self.features])
        cloud_predictions = self.predict(cloud_model, self.X_train[self.features])
        return edge_predictions, cloud_predictions


In [ ]:
testtese = dataPipelineDinges(train, test)
edge_predictions, cloud_predictions = testtese.run()

ValueError: could not convert string to float: 'W'

# Eerste test onzin

Spul hieronder was gewoon automatisch door copilot gemaakt, maar misschien staat er iets nuttigs in?

In [ ]:


def edge_model():
    # Edge Model: Simple anomaly detection using IsolationForest

    # Select a few relevant numeric features for the edge model
    edge_features = ['TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2']
    X_edge = transaction_train[edge_features].fillna(-999)

    # Train IsolationForest as a lightweight anomaly detector
    edge_detector = IsolationForest(n_estimators=50, contamination=0.01, random_state=42)
    edge_detector.fit(X_edge)

    # Predict anomalies on new transactions (simulate edge device)
    X_test_edge = transaction_test[edge_features].fillna(-999)
    edge_preds = edge_detector.predict(X_test_edge)  # -1 = anomaly, 1 = normal

    # Mark suspicious transactions for cloud validation
    transaction_test['edge_flag'] = edge_preds
    suspect_fraud = transaction_test[transaction_test['edge_flag'] == -1]

    # Simulate API call: send suspicious transactions to cloud model
    # (Here, just print how many would be sent)
    print(f"Number of transactions flagged as suspicious by edge model: {len(suspect_fraud)}")
    return suspect_fraud

Number of transactions flagged as suspicious by edge model: 7426


# Cloud Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Select features and target for the cloud model
X = transaction_train.drop(columns=['isFraud', 'TransactionID'])
y = transaction_train['isFraud']

# Split data for training and validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Build a simple deep learning model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train_scaled, y_train, epochs=5, batch_size=256, validation_data=(X_val_scaled, y_val))


# Example: Predict on suspicious transactions (from edge)
X_suspicious = suspect_fraud[cloud_features].fillna(-999)
X_suspicious_scaled = scaler.transform(X_suspicious)
cloud_preds = model.predict(X_suspicious_scaled)
suspicious['cloud_fraud_prob'] = cloud_preds

# Mark as fraud if probability > 0.5
suspicious['cloud_flag'] = (suspicious['cloud_fraud_prob'] > 0.5).astype(int)

print(suspicious[['TransactionID', 'cloud_fraud_prob', 'cloud_flag']].head())




Epoch 1/5


1846/1846 [==============================] - 4s 2ms/step - loss: 0.1562 - accuracy: 0.9636 - val_loss: 0.1392 - val_accuracy: 0.9652
Epoch 2/5
1846/1846 [==============================] - 3s 2ms/step - loss: 0.1431 - accuracy: 0.9651 - val_loss: 0.1389 - val_accuracy: 0.9653
Epoch 3/5
1846/1846 [==============================] - 3s 2ms/step - loss: 0.1417 - accuracy: 0.9652 - val_loss: 0.1384 - val_accuracy: 0.9652
Epoch 4/5
1846/1846 [==============================] - 3s 2ms/step - loss: 0.1406 - accuracy: 0.9652 - val_loss: 0.1380 - val_accuracy: 0.9653
Epoch 5/5
233/233 [==============================] - 0s 647us/step
     TransactionID  cloud_fraud_prob  cloud_flag
309        3663858          0.030923           0
360        3663909          0.208107           0
471        3664020          0.113161           0
687        3664236          0.065208           0
879        3664428          0.029639           0


C:\Users\tijnw\AppData\Local\Temp\ipykernel_31496\2470372096.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suspicious['cloud_fraud_prob'] = cloud_preds
C:\Users\tijnw\AppData\Local\Temp\ipykernel_31496\2470372096.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suspicious['cloud_flag'] = (suspicious['cloud_fraud_prob'] > 0.5).astype(int)
